# Notebook 0 — Loading PETR4.SA and Diagnosing Data Quality

<div style="background:rgba(66,133,244,0.08);border-left:5px solid #4285f4;border-radius:6px;margin:12px 0;overflow:hidden;"><div style="border-bottom:1px solid rgba(66,133,244,0.3);text-align:center;padding:6px 12px;font-weight:bold;font-size:1em;color:#4285f4;">Coverage</div><div style="padding:10px 14px;color:inherit;font-size:0.97em;">This notebook establishes a reproducible pipeline for acquiring, caching, and quality-assessing 26 years of daily OHLCV price data for Petrobras (PETR4.SA), and packages that workflow into a reusable <code>DataLoader</code> class that every subsequent notebook in the series depends on.</div></div>

<div style="background:rgba(52,168,83,0.08);border-left:5px solid #34a853;border-radius:6px;margin:12px 0;overflow:hidden;"><div style="border-bottom:1px solid rgba(52,168,83,0.3);text-align:center;padding:6px 12px;font-weight:bold;font-size:1em;color:#34a853;">Dataset</div><div style="padding:10px 14px;color:inherit;font-size:0.97em;">Petrobras preferred shares (ticker <code>PETR4.SA</code>) traded on B3 (the Brazilian exchange). The series spans 2000–2026 at daily frequency, uniformly sampled by trading day, and exhibits the full range of financial time-series pathologies: non-stationarity, fat tails, volatility clustering, and regime shifts.</div></div>

<!-- intro:auto -->
This notebook builds a reproducible pipeline for acquiring, caching, and quality-assessing 26 years of daily OHLCV data for Petrobras (PETR4.SA), and packages that workflow into a reusable `DataLoader` class.

<div style="font-family:sans-serif;margin:20px 0 8px 0;max-width:600px;">

  <!-- Row 1 -->
  <div style="display:flex;align-items:flex-start;">
    <div style="display:flex;flex-direction:column;align-items:center;width:36px;flex-shrink:0;">
      <div style="width:32px;height:32px;border-radius:50%;border:2px solid #4285f4;background:rgba(66,133,244,0.08);display:flex;align-items:center;justify-content:center;font-weight:700;font-size:0.85em;color:#4285f4;">1</div>
      <div style="width:2px;background:rgba(128,128,128,0.25);flex:1;min-height:36px;"></div>
    </div>
    <div style="margin-left:14px;padding-bottom:20px;">
      <div style="font-weight:700;color:#4285f4;font-size:0.93em;">Loading the data</div>
      <div style="color:inherit;opacity:0.7;font-size:0.84em;margin-top:3px;">Fetch PETR4.SA OHLCV from yfinance with a reproducible local CSV cache.</div>
    </div>
  </div>

  <!-- Row 2 -->
  <div style="display:flex;align-items:flex-start;">
    <div style="display:flex;flex-direction:column;align-items:center;width:36px;flex-shrink:0;">
      <div style="width:32px;height:32px;border-radius:50%;border:2px solid #34a853;background:rgba(52,168,83,0.08);display:flex;align-items:center;justify-content:center;font-weight:700;font-size:0.85em;color:#34a853;">2</div>
      <div style="width:2px;background:rgba(128,128,128,0.25);flex:1;min-height:36px;"></div>
    </div>
    <div style="margin-left:14px;padding-bottom:20px;">
      <div style="font-weight:700;color:#34a853;font-size:0.93em;">Format and integrity inspection</div>
      <div style="color:inherit;opacity:0.7;font-size:0.84em;margin-top:3px;">Run seven sanity checks on index, dtypes, OHLC constraints, and missing values.</div>
      <div style="margin-top:6px;display:flex;flex-wrap:wrap;gap:5px;">
        <span style="font-size:0.75em;border:1px solid #34a853;border-radius:12px;padding:1px 9px;color:#34a853;white-space:nowrap;">Check 1 Index type &amp; monotonicity</span>
        <span style="font-size:0.75em;border:1px solid #34a853;border-radius:12px;padding:1px 9px;color:#34a853;white-space:nowrap;">Check 2 Date range</span>
        <span style="font-size:0.75em;border:1px solid #34a853;border-radius:12px;padding:1px 9px;color:#34a853;white-space:nowrap;">Check 3 Duplicates</span>
        <span style="font-size:0.75em;border:1px solid #34a853;border-radius:12px;padding:1px 9px;color:#34a853;white-space:nowrap;">Check 4 Dtypes</span>
        <span style="font-size:0.75em;border:1px solid #34a853;border-radius:12px;padding:1px 9px;color:#34a853;white-space:nowrap;">Check 5 OHLC sanity</span>
        <span style="font-size:0.75em;border:1px solid #34a853;border-radius:12px;padding:1px 9px;color:#34a853;white-space:nowrap;">Check 6 Non-negativity</span>
        <span style="font-size:0.75em;border:1px solid #34a853;border-radius:12px;padding:1px 9px;color:#34a853;white-space:nowrap;">Check 7 Missing values</span>
      </div>
    </div>
  </div>

  <!-- Row 3 -->
  <div style="display:flex;align-items:flex-start;">
    <div style="display:flex;flex-direction:column;align-items:center;width:36px;flex-shrink:0;">
      <div style="width:32px;height:32px;border-radius:50%;border:2px solid #e37400;background:rgba(227,116,0,0.08);display:flex;align-items:center;justify-content:center;font-weight:700;font-size:0.85em;color:#e37400;">3</div>
      <div style="width:2px;background:rgba(128,128,128,0.25);flex:1;min-height:36px;"></div>
    </div>
    <div style="margin-left:14px;padding-bottom:20px;">
      <div style="font-weight:700;color:#e37400;font-size:0.93em;">DataLoader class</div>
      <div style="color:inherit;opacity:0.7;font-size:0.84em;margin-top:3px;">Wrap fetch and inspect into a reusable class for all downstream notebooks.</div>
    </div>
  </div>

  <!-- Row 4 (no connector below last badge) -->
  <div style="display:flex;align-items:flex-start;">
    <div style="display:flex;flex-direction:column;align-items:center;width:36px;flex-shrink:0;">
      <div style="width:32px;height:32px;border-radius:50%;border:2px solid #8430ce;background:rgba(132,48,206,0.08);display:flex;align-items:center;justify-content:center;font-weight:700;font-size:0.85em;color:#8430ce;">4</div>
    </div>
    <div style="margin-left:14px;">
      <div style="font-weight:700;color:#8430ce;font-size:0.93em;">Act 4 — Conclusions</div>
      <div style="color:inherit;opacity:0.7;font-size:0.84em;margin-top:3px;">Headline findings from the loading and integrity workflow.</div>
    </div>
  </div>

</div>

In [ ]:
# !pip install -r requirements.txt

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mplfinance as mpf
import yfinance as yf

from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf

warnings.filterwarnings("ignore", category=FutureWarning)

plt.style.use("dark_background")
_BG = "#000000"
sns.set_theme(context="notebook", style="darkgrid", rc={
    "axes.facecolor": _BG,
    "figure.facecolor": _BG,
    "savefig.facecolor": _BG,
    "axes.edgecolor": "#cccccc",
    "axes.labelcolor": "#eeeeee",
    "axes.titlecolor": "#eeeeee",
    "xtick.color": "#cccccc",
    "ytick.color": "#cccccc",
    "text.color": "#eeeeee",
    "grid.color": "#222222",
    "legend.facecolor": "#111111",
    "legend.edgecolor": "#333333",
    "patch.edgecolor": "#cccccc",
})
plt.rcParams["figure.figsize"] = (11, 4)

import plotly.io as pio
_tpl = pio.templates["plotly_dark"]
_tpl.layout.paper_bgcolor = _BG
_tpl.layout.plot_bgcolor = _BG
pio.templates["black_bg"] = _tpl
pio.templates.default = "black_bg"
MPF_STYLE = mpf.make_mpf_style(
    base_mpf_style="nightclouds",
    facecolor=_BG, edgecolor=_BG, figcolor=_BG,
    gridcolor="#222222", gridstyle=":",
    rc={"axes.labelcolor": "#eeeeee", "xtick.color": "#cccccc", "ytick.color": "#cccccc"},
)
np.random.seed(42)


In [2]:
TICKER = "PETR4.SA"
END_DATE = "2026-05-26"
DATA_DIR = Path("data")
CSV_PATH = DATA_DIR / "PETR4_SA.csv"
DATA_DIR.mkdir(exist_ok=True)

## Act 1 — Loading the data (with a local cache)

We use `yfinance` to download the entire price history of `PETR4.SA` up to and including 2026-05-26. To make the notebook **reproducible**, we cache the raw download as a CSV the first time we run it and reload from disk thereafter.

We pass `auto_adjust=False` so the dataframe keeps both `Close` (raw) and `Adj Close` (adjusted for splits and dividends) — useful later when we discuss which series to model.

In [3]:
if CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH, index_col="Date", parse_dates=True)
    print(f"Loaded from cache: {CSV_PATH}  ({len(df):,} rows)")
else:
    df = yf.download(TICKER, end=END_DATE, period="max", auto_adjust=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.index.name = "Date"
    df.to_csv(CSV_PATH)
    print(f"Downloaded and cached to {CSV_PATH}  ({len(df):,} rows)")

Loaded from cache: data\PETR4_SA.csv  (6,624 rows)


In [4]:
df.head()

,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2000-01-03,1.105877,5.875,5.875,5.875,5.875,35389440000
2000-01-04,1.044701,5.550,5.550,5.550,5.550,28861440000
2000-01-05,1.034160,5.494,5.494,5.494,5.494,43033600000
2000-01-06,1.030583,5.475,5.475,5.475,5.475,34055680000
2000-01-07,1.035289,5.500,5.500,5.500,5.500,20912640000


In [5]:
df.tail()

,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2026-05-19,46.090000,46.090000,46.299999,45.590000,45.990002,40943000
2026-05-20,44.599998,44.599998,46.410000,44.540001,45.770000,52780200
2026-05-21,44.950001,44.950001,45.650002,44.500000,45.099998,57725900
2026-05-22,44.480000,44.480000,44.750000,43.869999,44.740002,39634300
2026-05-25,43.400002,43.400002,43.820000,42.970001,43.490002,26720900


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 6624 entries, 2000-01-03 to 2026-05-25
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Adj Close  6624 non-null   float64
 1   Close      6624 non-null   float64
 2   High       6624 non-null   float64
 3   Low        6624 non-null   float64
 4   Open       6624 non-null   float64
 5   Volume     6624 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 362.2 KB


In [7]:
print('Shape:', df.shape)
print('Index type:', type(df.index).__name__)
print('Dtypes:')
print(df.dtypes)

Shape: (6624, 6)
Index type: DatetimeIndex
Dtypes:
Adj Close    float64
Close        float64
High         float64
Low          float64
Open         float64
Volume         int64
dtype: object


<div style="background:rgba(249,171,0,0.08);border-left:5px solid #f9ab00;border-radius:6px;margin:12px 0;overflow:hidden;"><div style="border-bottom:1px solid rgba(249,171,0,0.3);text-align:center;padding:6px 12px;font-weight:bold;font-size:1em;color:#f9ab00;">What We See</div><div style="padding:10px 14px;color:inherit;font-size:0.97em;">The dataframe spans from PETR4.SA&#39;s earliest available trading day up to 2026-05-26. Six columns are returned by yfinance: <code>Open</code>, <code>High</code>, <code>Low</code>, <code>Close</code>, <code>Adj Close</code>, <code>Volume</code>. The index is a <code>DatetimeIndex</code> of trading days. All price columns are <code>float64</code>; <code>Volume</code> is integer-typed but large enough to overflow <code>int32</code> on some platforms — pandas usually picks <code>int64</code>.</div></div>

## Act 2 — Format and integrity inspection

Before any analysis, verify the data is what we think it is. We run seven checks in order: index type, date range, duplicate timestamps, dtypes, OHLC sanity, non-negativity, and missing values.

### Check 1 — Index type and monotonicity

In [8]:
assert isinstance(df.index, pd.DatetimeIndex), "Index must be a DatetimeIndex"
print("Monotonic increasing:", df.index.is_monotonic_increasing)
print("Unique index values:", df.index.is_unique)

Monotonic increasing: True
Unique index values: True


### Check 2 — Date range coverage

In [9]:
print("First date :", df.index.min().date())
print("Last  date :", df.index.max().date())
print("Total rows :", len(df))
print("Span (yrs) :", round((df.index.max() - df.index.min()).days / 365.25, 2))

First date : 2000-01-03
Last  date : 2026-05-25
Total rows : 6624
Span (yrs) : 26.39


### Check 3 — Duplicate timestamps

In [10]:
print("Duplicate index entries:", int(df.index.duplicated().sum()))

Duplicate index entries: 0


### Check 4 — Column dtypes

In [11]:
price_cols = ["Open", "High", "Low", "Close", "Adj Close"]
for c in price_cols:
    assert pd.api.types.is_float_dtype(df[c]), f"{c} should be float"
assert pd.api.types.is_integer_dtype(df["Volume"]) or pd.api.types.is_float_dtype(df["Volume"])
print("All numeric dtypes OK")

All numeric dtypes OK


### Check 5 — OHLC sanity

In [13]:
checks = {
    "High >= Low":   (df["High"] >= df["Low"]).all(),
    # Open in the range
    "Open >= Low":   (df["Open"] >= df["Low"]).all(),
    "Open <= High":  (df["Open"] <= df["High"]).all(),
    # Close in the range
    "Close >= Low":  (df["Close"] >= df["Low"]).all(),
    "Close <= High": (df["Close"] <= df["High"]).all(),
}

for k, v in checks.items():
    print(f"{k}: {v}")

High >= Low: True
Open >= Low: True
Open <= High: True
Close >= Low: False
Close <= High: False


In [14]:
invalid = df[
    (df["High"] < df["Low"]) |
    (df["Open"] < df["Low"]) |
    (df["Open"] > df["High"]) |
    (df["Close"] < df["Low"]) |
    (df["Close"] > df["High"])
]

print(invalid)

            Adj Close      Close       High        Low       Open     Volume
Date                                                                        
2006-05-10   4.683244  24.075001  24.049999  23.760000  24.000000  470056352
2006-05-12   4.606406  23.680000  23.600000  23.250000  23.325001  651535232
2006-05-26   4.224160  21.715000  22.549999  21.775000  22.000000  658253824
2006-05-30   4.352548  22.375000  22.299999  21.625000  22.200001  596815424
2006-06-16   3.705743  19.049999  20.365000  19.475000  19.805000  641583552
2006-06-28   3.949876  20.305000  20.715000  20.400000  20.400000  391424544
2006-06-29   4.026714  20.700001  21.549999  20.799999  20.815001  627573632
2006-07-05   4.222215  21.705000  21.615000  21.055000  21.400000  546497024
2006-07-06   4.162883  21.400000  21.700001  21.424999  21.549999  562186752
2006-07-13   4.261121  21.905001  21.900000  21.600000  21.750000  398954624
2006-07-14   4.240695  21.799999  22.160000  21.850000  21.885000  550678592

<div style="background:rgba(147,52,230,0.08);border-left:5px solid #9334e6;border-radius:6px;margin:12px 0;overflow:hidden;"><div style="border-bottom:1px solid rgba(147,52,230,0.3);text-align:center;padding:6px 12px;font-weight:bold;font-size:1em;color:#9334e6;">Yahoo Finance Data Corruption — Root Cause</div><div style="padding:10px 14px;color:inherit;font-size:0.97em;">Yahoo Finance changed its data format in 2017 to return <strong>split-adjusted OHLC values</strong> while simultaneously returning <strong>unadjusted dividend data</strong>, creating an inconsistency in how Adjusted Close is calculated. Yahoo mixes pre-split dividends with already-split-adjusted prices, producing <strong>over-modification</strong> that makes Adj Close far smaller than Close. In this dataset it manifests as OHLC values around $20–24 while Adj Close is only $4–5 (a ~5.5&times; ratio indicating a 2014 stock split), plus impossible OHLC violations like High &lt; Low. The fix is to pass <code>auto_adjust=True</code> to <code>yf.download</code>. <a href="https://github.com/joshuaulrich/quantmod/issues/253" target="_blank">See GitHub issue.</a></div></div>

### Check 6 — Non-negativity

In [15]:
print("Volume >= 0:", (df["Volume"] >= 0).all())
print("Prices  > 0:", (df[price_cols] > 0).all().all())

Volume >= 0: True
Prices  > 0: True


### Check 7 — Missing values

In [16]:
na_counts = df.isna().sum()
print(na_counts)
if na_counts.sum() == 0:
    print("\nNo missing values detected — skipping imputation discussion (see Ch6).")
else:
    print("\nMissing values present — see Ch6 for imputation strategies.")

Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64

No missing values detected — skipping imputation discussion (see Ch6).


The data passes all seven checks. Note that weekends and Brazilian holidays produce *gaps* in the date index, which is **expected and not the same as missing data** — those days simply did not exist as trading days. We move on without imputation.

## Act 3 — Putting it together: a `DataLoader` class

Acts 1 and 2 are workflow steps repeated in every notebook in this series. The loading and inspection logic is packaged into a small `DataLoader` class to keep downstream notebooks DRY. Statistical exploration is deferred to Notebook 1, where it belongs by nature.


In [17]:
class DataLoader:
    """Cache-aware loader for daily OHLCV data via yfinance."""

    def __init__(self, ticker: str, end: str, cache_dir: Path = Path("data")):
        self.ticker = ticker
        self.end = end
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(exist_ok=True)
        self.csv_path = self.cache_dir / f"{ticker.replace('.', '_')}.csv"
        self.df: pd.DataFrame | None = None

    def fetch(self) -> pd.DataFrame:
        if self.csv_path.exists():
            self.df = pd.read_csv(self.csv_path, index_col="Date", parse_dates=True)
        else:
            self.df = yf.download(self.ticker, end=self.end, period="max",
                                  auto_adjust=False)
            if isinstance(self.df.columns, pd.MultiIndex):
                self.df.columns = self.df.columns.get_level_values(0)
            self.df.index.name = "Date"
            self.df.to_csv(self.csv_path)
        return self.df

    def load(self) -> pd.DataFrame:
        if not self.csv_path.exists():
            raise FileNotFoundError(f"No cache at {self.csv_path}; call fetch() first.")
        self.df = pd.read_csv(self.csv_path, index_col="Date", parse_dates=True)
        return self.df

    def inspect(self) -> dict:
        if self.df is None:
            raise RuntimeError("Call fetch() or load() first.")
        d = self.df
        return {
            "rows":            len(d),
            "first_date":      d.index.min().date().isoformat(),
            "last_date":       d.index.max().date().isoformat(),
            "monotonic":       bool(d.index.is_monotonic_increasing),
            "unique_index":    bool(d.index.is_unique),
            "duplicates":      int(d.index.duplicated().sum()),
            "na_total":        int(d.isna().sum().sum()),
            "ohlc_consistent": bool(
                (d["High"] >= d["Low"]).all()
                and (d["High"] >= d["Close"]).all()
                and (d["Low"]  <= d["Open"]).all()
            ),
        }

    def run(self) -> pd.DataFrame:
        self.fetch()
        report = self.inspect()
        print("Inspection report:")
        for k, v in report.items():
            print(f"  {k:<18} {v}")
        return self.df

In [18]:
loader = DataLoader("PETR4.SA", end="2026-05-26")
df2 = loader.run()
df2.head()

Inspection report:
  rows               6624
  first_date         2000-01-03
  last_date          2026-05-25
  monotonic          True
  unique_index       True
  duplicates         0
  na_total           0
  ohlc_consistent    False


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2000-01-03,1.105877,5.875,5.875,5.875,5.875,35389440000
2000-01-04,1.044701,5.550,5.550,5.550,5.550,28861440000
2000-01-05,1.034160,5.494,5.494,5.494,5.494,43033600000
2000-01-06,1.030583,5.475,5.475,5.475,5.475,34055680000
2000-01-07,1.035289,5.500,5.500,5.500,5.500,20912640000


<div style="background:rgba(66,133,244,0.08);border-left:5px solid #4285f4;border-radius:6px;margin:12px 0;overflow:hidden;"><div style="border-bottom:1px solid rgba(66,133,244,0.3);text-align:center;padding:6px 12px;font-weight:bold;font-size:1em;color:#4285f4;">Why Refactor?</div><div style="padding:10px 14px;color:inherit;font-size:0.97em;">Every subsequent notebook in this series runs the same fetch-and-inspect ritual. The <code>DataLoader</code> class reduces that to a single call — <code>DataLoader(ticker, end).run()</code> — and centralises any future fix (e.g., a yfinance API change) in one place. The class is defined inline here for self-containment; a later notebook promotes it to a shared <code>utils.py</code> module.</div></div>

<!-- conclusion:auto -->
## Act 4 — Conclusions

- **Act 1 — Loading the data (with a local cache)** — `yf.download` with `auto_adjust=False` delivers 6,624 daily OHLCV rows for PETR4.SA spanning 2000-01-03 to 2026-05-25 (≈26.4 years); a CSV cache makes every re-run offline and bit-for-bit reproducible.

- **Act 2 — Format and integrity inspection** — The index is a monotonic, unique `DatetimeIndex` with zero duplicates and zero missing values; however, 20 rows fail OHLC sanity because Yahoo Finance mixes split-adjusted OHLC with unadjusted dividend data, producing a ~5.5× ratio between `Close` and `Adj Close` — resolved by passing `auto_adjust=True`.

- **Act 3 — Putting it together: a `DataLoader` class** — Wrapping fetch-or-load and the seven integrity checks behind `DataLoader.run()` confirms the same findings in a single call and provides a reusable entry point for every notebook that follows.